In [ ]:
环境安装安装魔搭提供的SDK，什么是SDK
SDK (Software Development Kit) - 软件开发工具包
一、基本概念
SDK 是一组软件开发工具的集合，帮助开发者快速为特定平台、框架或服务创建应用程序。

简单来说：

API 是接口（如何调用）

SDK 是工具包（包含API + 开发工具 + 文档 + 示例代码）


In [ ]:
pip install modelscope

In [ ]:
使用魔搭SDK提供的snapshot_download方法进行高速下载模型，下载好的模型会存储在result文件夹下

In [ ]:
from modelscope import snapshot_download
model_dir = snapshot_download("Qwen/Qwen2.5-0.5B-Instruct")

In [28]:
#验证是否安装了transformer
import sys
import torch
import transformers
import modelscope
from modelscope import snapshot_download

print("Python 版本:", sys.version.split()[0])
print("PyTorch 版本:", torch.__version__)
print("Transformers 版本:", transformers.__version__)



Python 版本: 3.11.11
PyTorch 版本: 2.9.1+cpu
Transformers 版本: 5.3.0


In [ ]:
#下载langchain
!pip install langchain

In [ ]:
#使用langchain来搭建一个RAG知识问答机器人
#首先进行文档清洗，文档可能包含多种类型，PDF，CSV，TXT等等这里使用到langchian的lPyPDFLoader（）方法
# 安装 PyPDFLoader 和相关依赖
!pip install langchain-community pypdf

In [21]:
# 检查 PDF 是否包含文本
import PyPDF2

with open("pdf/Linux高性能服务器编程.pdf", "rb") as f:
    reader = PyPDF2.PdfReader(f)
    first_page = reader.pages[0]
    text = first_page.extract_text()
    
    if not text.strip():
        print("❌ 该 PDF 是扫描版或图片版，没有文字层")
        print("   需要 OCR 识别才能提取文字")
    else:
        print(f"✅ 有文字层，长度: {len(text)}")

❌ 该 PDF 是扫描版或图片版，没有文字层
   需要 OCR 识别才能提取文字


In [ ]:
pip install transformers

In [ ]:
pip install PyMuPDF

In [39]:
#使用PyMuPDF来提取扫描件文字
import fitz  # PyMuPDF

def extract_text_with_fitz(pdf_path):
    doc = fitz.open(pdf_path)
    full_text = []
    for page in doc:
        # 提取当前页的所有文本
        full_text.append(page.get_text("text"))
    doc.close()
    return "\n".join(full_text)

text = extract_text_with_fitz("pdf/人工智能通识教程.pdf")
print(text[:500])  # 打印前500个字符


人工智能通识教程
北　京
主　编　程国安　刘隆吉　蔡兴壮
副主编　盖延亮　王家波　
高等院校产教融合创新应用系列

内 容 简 介
本书主要介绍人工智能的核心技术与应用发展，内容涵盖人工智能概述、人工智能技术原理、人工智
能交叉学科及融合技术、人工智能应用方向、人工智能应用场景、人工智能发展展望和人工智能时代的职
业展望。引入丰富的实践案例和拓展阅读，旨在帮助学生深入理解人工智能的技术脉络与社会影响，掌握
相关领域的基础知识，提升技术应用能力，适应数字化时代的发展需求。
本书结构清晰，理论与实践结合，可作为职业本科院校、高职高专院校人工智能通识课程的教材，也
可作为对人工智能技术感兴趣的读者的参考资料。
本书封面贴有清华大学出版社防伪标签，无标签者不得销售。
版权所有，侵权必究。举报：010-62782989，beiqinquan@tup.tsinghua.edu.cn。
图书在版编目(CIP) 数据
人工智能通识教程 / 程国安, 刘隆吉, 蔡兴壮主编.
北京 : 清华大学出版社, 2025. 8. -- ( 高等院校产教融合
创新应用系列). -- ISBN 978-7-302-70080-7 
Ⅰ. TP18 
中国国家版本馆CIP 数据核字第202528523T 号
责任编辑：王　定
封面设计：周晓亮
版式设计：思创景点
责任校对：马遥遥 
责任印制：宋　林
出版发行：清华大学出版社
网　　　址：https://www.tup.com.cn，https://www.wqxuetang.com
地　　　址：北京清华大学学研大厦A座	
	
邮　　编：100084
社   总    机：010-83470000	
	
	
邮　　购：010-62786544
投稿与读者服务：010-62776969，c-service@tup.tsinghua.edu.cn
质　量　反　馈：010-62772015，zhiliang@tup.tsinghua.edu.cn
印  装  者：三河市龙大印装有限公司
经　　销：全国新华书店
开　　本：185mm×260mm　　　　	
印　　张：11.75	 	
字　　数：264千字
版　　次：2025 年9月第 1 版	
印　　次：2025 年9月第 1 次印刷
定　　价：49.80元
—————————————————————————

In [50]:
import fitz
import torch
from modelscope import AutoTokenizer, AutoModel, pipeline
from modelscope.utils.constant import Tasks
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_modelscope import ModelScopeEmbeddings  # 替代 HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [ ]:
pip install langchain-modelscope-integration

In [52]:
import fitz
from langchain_core.documents import Document
# 转换为 LangChain Document 对象
documents = [Document(page_content=text, metadata={"source": "人工智能通识教程.pdf"})]

print(f"文档对象数量: {len(documents)}")
print(f"文档内容长度: {len(documents[0].page_content)} 字符")

文档对象数量: 1
文档内容长度: 29794 字符


In [ ]:
!pip install langchain-huggingface

In [57]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 创建分割器
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # 每块500字符
    chunk_overlap=50,    # 重叠50字符
)

# 分块（documents 是列表）
chunks = splitter.split_documents(documents)

print(f"分成了 {len(chunks)} 块")

分成了 80 块


In [63]:
import fitz
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from transformers import pipeline as hf_pipeline

In [ ]:
import os
from sentence_transformers import SentenceTransformer
from langchain.embeddings.base import Embeddings
from typing import List

# 设置镜像（加速下载）
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

class BGEEmbeddings(Embeddings):
    """BGE 中文嵌入模型 - 无 onnx 依赖问题"""
    
    def __init__(self, model_name: str = "BAAI/bge-small-zh-v1.5"):
        print(f"加载模型: {model_name}")
        self.model = SentenceTransformer(model_name)
        print(f"✓ 模型加载完成，维度: {self.model.get_sentence_embedding_dimension()}")
    
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return self.model.encode(texts, normalize_embeddings=True).tolist()
    
    def embed_query(self, text: str) -> List[float]:
        return self.model.encode(text, normalize_embeddings=True).tolist()

# 使用
embeddings = BGEEmbeddings("BAAI/bge-small-zh-v1.5")

In [ ]:
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [68]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA

os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

# ==================== 1. 加载数据 ====================
loader = PyMuPDFLoader("pdf/人工智能通识教程.pdf")
documents = loader.load()

# ==================== 2. 分块 ====================
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

# ==================== 3. 向量存储 ====================
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# ==================== 4. 加载 LLM (使用 AutoModel) ====================
print("加载语言模型...")

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

# 加载分词器和模型
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
    low_cpu_mem_usage=True
)

if device == "cpu":
    model = model.to(device)

# 创建 pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.7,
    top_p=0.9,
    do_sample=True
)

# 包装为 LangChain LLM
llm = HuggingFacePipeline(pipeline=pipe)

# ==================== 5. 创建 RAG 链 ====================
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    verbose=True  # 打印详细过程
)

# ==================== 6. 交互问答 ====================
print("\n✓ RAG系统初始化完成！")
while True:
    q = input("\n问题: ").strip()
    if q.lower() == 'exit':
        break
    if not q:
        continue
    
    try:
        answer = qa_chain.invoke(q)
        print(f"回答: {answer['result']}")
    except Exception as e:
        print(f"错误: {e}")

ModuleNotFoundError: No module named 'langchain.document_loaders'